<a href="https://colab.research.google.com/github/andreamarin/senate-publications-analysis/blob/add%2Fbertopic-analysis/bertopic_modeling_news.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Set up

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
!git clone https://github.com/andreamarin/senate-publications-analysis.git

Cloning into 'senate-publications-analysis'...
remote: Enumerating objects: 587, done.
remote: Counting objects: 100% (146/146), done.
remote: Compressing objects: 100% (114/114), done.
remote: Total 587 (delta 80), reused 59 (delta 32), pack-reused 441 (from 2)
Receiving objects: 100% (587/587), 2.34 MiB | 21.18 MiB/s, done.
Resolving deltas: 100% (364/364), done.


In [3]:
%cd senate-publications-analysis/nlp_classification/

/content/senate-publications-analysis/nlp_classification


In [4]:
!git checkout add/bertopic-analysis

Branch 'add/bertopic-analysis' set up to track remote branch 'add/bertopic-analysis' from 'origin'.
Switched to a new branch 'add/bertopic-analysis'


In [5]:
%mkdir config

In [6]:
%cp ../../drive/MyDrive/tesis/code/config/* ./config/.

In [7]:
%ls config

bot-cert.pem


In [8]:
!git pull

Already up to date.


In [9]:
!pip install -r bert_requirements.txt

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.7/154.7 kB 17.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.8/27.8 MB 98.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 83.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 512.4/512.4 kB 52.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.8/88.8 kB 12.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.6/2.6 MB 109.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.9/79.9 MB 28.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 331.1/331.1 kB 37.6 MB/s eta 0:00:00
  Attempting uninstall: umap-learn
    Found existing installation: umap-learn 0.5.12
    Uninstalling umap-learn-0.5.12:
      Successfully uninstalled umap-learn-0.5.12
  Attempting uninstall: sentence-transformers
    Found existing installation: sentence-transformers 5.7.0
    Uninstalling sentence-transformers-5.7.0:
      Successfully uninstalled sentence-t

In [10]:
! python -m spacy download es_core_news_lg

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 568.0/568.0 MB 3.5 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('es_core_news_lg')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [11]:
!curl ipecho.net/plain

136.110.61.138

In [12]:
from psutil import virtual_memory
ram_gb = virtual_memory().total / 1e9
print('Your runtime has {:.1f} gigabytes of available RAM\n'.format(ram_gb))

if ram_gb < 20:
  print('Not using a high-RAM runtime')
else:
  print('You are using a high-RAM runtime!')

Your runtime has 179.4 gigabytes of available RAM

You are using a high-RAM runtime!


# Imports

In [13]:
import sys
sys.path.append('/content/senate-publications-analysis/nlp_classification')

In [14]:
import warnings
warnings.filterwarnings('ignore')

In [15]:
import os
import re
import nltk
import importlib
import pandas as pd
from datetime import datetime
from tqdm import tqdm

In [16]:
import utils.db as db
import utils.nlp_processor as nlp
import utils.bertopic_model_builder as model_builder
import utils.bertopic_evaluator as model_evaluator
from utils.bertopic_config import (
    EmbeddingConfig,
    UMAPConfig,
    HDBSCANConfig,
    DocumentRepresentation,
    ComputeConfig,
    OutlierReductionConfig,
    OutlierReductionStrategy,
    CountVectorizerConfig,
    EvaluationConfig,
)
from utils.bertopic_results_comparator import generate_metrics_comparison_graphs
from utils.hierarchy_merge_parser import build_merge_groups_from_tree, build_merge_topic_list_from_file


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [19]:
import warnings
warnings.filterwarnings('ignore')

In [18]:
nltk.download('stopwords')

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


True

In [ ]:
importlib.reload(model_builder)

# Load data

In [20]:
conn = db.connect_mongo_db("news-data")

In [23]:
articles_cursor = conn.articles.find(
    projection=["newspaper", "curated_section", "date", "text", "summary", "updated", "clean_text"]
  )
articles_df = pd.DataFrame(articles_cursor)
articles_df.head()

,_id,newspaper,date,summary,text,updated,curated_section,clean_text
0,330980da03e75b9dabfb2970032a60d3,Proceso,2018-01-30T00:00:00,,CIUDAD DE MÉXICO (apro).- La Comisión Federal ...,2026-06-10 06:34:18.817,economia,la comisión federal de competencia económica (...
1,bc9c780c3d54918b0d52abd584424a8d,Proceso,2018-01-29T00:00:00,,CIUDAD DE MÉXICO (apro).- La sexta ronda de re...,2026-06-10 06:34:18.817,economia,la sexta ronda de renegociación del tratado de...
2,84bbad08a55de5c9cbf539ee0bbde843,Proceso,2018-01-17T00:00:00,,CIUDAD DE MÉXICO (apro).- La francesa Total ab...,2026-06-10 06:34:18.817,economia,la francesa total abrió este miércoles 17 la p...
3,f1601a6e0676a3674d29336ed568a7b0,Proceso,2018-01-27T00:00:00,,CIUDAD DE MÉXICO (Proceso).- En medio de numer...,2026-06-10 06:34:18.817,opinion,en medio de numerosas opiniones sobre el futur...
4,24088e2b3760b624095941830fd2667e,Proceso,2018-01-24T00:00:00,,CIUDAD DE MÉXICO (apro).--La inflación inició ...,2026-06-10 06:34:18.817,economia,"-la inflación inició el año a la baja, sin emb..."


In [24]:
remove_sections = [
    "espectaculos",
    "tendencias",
    "estilo",
    "entretenimiento",
    "autos",
    "food and drink",
    "encuestas",
    "algarabia",
    "retrato hablado",
    "signos y senales",
    "rankings",
    "finanzas personales",
    "after office",
    "transicion",
    "rankings",
    "inmobiliario",
    "emprendedores",
    "management",
    "viajes",
    "el preguntario",
    "el empresario",
    "editorial",
    "hablemos de",
    "empresas",
    "millonarios"
]

In [25]:
# 447_072
filtered_df = articles_df.loc[~articles_df.curated_section.isin(remove_sections)]
filtered_df.shape

(447072, 8)

# Clean text

In [ ]:
def remove_tags(text: str) -> str:
  return re.sub(r"<\/?.*?>", "", text)

def remove_news_start(text: str) -> str:
  """
  Remove the <city_name> (apro).- from the beginning of the text
  """
  return re.sub(r"^[\wáéíóúÁÉÍÓÚ\s,]+\.?-?\s?\((.*?)\)+\.?-?\s?", "", text)

In [ ]:
enabled_steps = {
    "extra_processing": True,
    "remove_words": False,
    "remove_punctuation": False,
    "lemmatize": False,
    "stop_words": False,
}
procesor = nlp.NlpProcessor(
    texts_df = articles_df,
    spacy_model_name = "es_core_news_lg",
    process_text_config = {
        "enabled_steps": enabled_steps,
    },
    extra_processing_steps = [remove_tags, remove_news_start],
)

In [ ]:
print(filtered_df.updated.max())
last_updated = filtered_df.updated.max()

In [ ]:
pending_articles_condition = (
    (filtered_df.updated <= last_updated)
)

pending_articles = filtered_df.loc[pending_articles_condition].reset_index(drop=True)

In [ ]:
total_artices = pending_articles.shape[0]
total_artices

In [ ]:
for start in tqdm(range(0, total_artices, BATCH_SIZE)):
  end = min(start + BATCH_SIZE, total_artices)

  batch_df = pending_articles.iloc[start:end]

  batch_df.loc[:, "clean_text"] = procesor.process_corpus(batch_df.text)

  # keep only the needed columns
  batch_df = batch_df[["_id", "clean_text"]]

  # add updated timestamp
  batch_df["updated"] = datetime.now()

  # update records in the DB
  db.batch_update_records(
      batch_df.to_dict(orient="records"),
      "articles",
      conn,
  )

# Run models

### Optional: install RAPIDS cuML (GPU UMAP/HDBSCAN)

Only needed once per Colab runtime. If this fails, the builder falls back to CPU `umap`/`hdbscan`.

```python
# CUDA 12 Colab example — adjust if your runtime uses a different CUDA version:
# %pip install cuml-cu12 --extra-index-url=https://pypi.nvidia.com
```


In [26]:
BASE_PATH = "/content/drive/MyDrive/tesis/bertopic_models"
FOLDER_NAME = "news"

In [27]:
base_params = {
    "texts_df": articles_df,
    "text_column": "clean_text",
    "folder_name": FOLDER_NAME,
    "base_path": BASE_PATH,
    "verbose": True,
    # Auto-uses cuML UMAP/HDBSCAN when installed; falls back to CPU otherwise.
    "use_cuml": True,
    "evaluation_config": EvaluationConfig(
        silhouette_sample_size=350_000,
        coherence_metrics=("c_v", "u_mass", "c_npmi"),
    ),
    "countvectorizer_config": CountVectorizerConfig(
        ngram_range=(1, 2),
        min_df=10,
        lowercase=True,
        strip_accents=None,
        extra_stop_words = ["apro"]
    ),
}


## Run different options

In [32]:
# Use the same model as the one used in the senate topic classification
EMBEDDING_MODEL = "sentence-transformers/paraphrase-multilingual-mpnet-base-v2"

document_representations = [
    DocumentRepresentation.CHUNKS,
    # DocumentRepresentation.MEAN_POOLING,
    DocumentRepresentation.FULL_TEXT,
    # DocumentRepresentation.MAX_POOLING,
]

EMBEDDING_CONFIGS = []
for dr in document_representations:
    EMBEDDING_CONFIGS.append(
        EmbeddingConfig(
            embedding_model=EMBEDDING_MODEL,
            max_words=80,
            spacy_model="es_core_news_lg",
            document_representation=dr,
            encode_batch_size=16384
        )
    )


HDBSCAN_CONFIGS = [
    HDBSCANConfig(min_cluster_size=20, prediction_data=True),
    HDBSCANConfig(min_cluster_size=25, prediction_data=True),
    HDBSCANConfig(min_cluster_size=30, prediction_data=True),
]

UMAP_CONFIGS = [
    UMAPConfig(n_neighbors=15, metric="cosine", min_dist=0.1),
    UMAPConfig(n_neighbors=20, metric="cosine", min_dist=0.1),
]

In [33]:
params_options = []
for emb_config in EMBEDDING_CONFIGS:
  for hdb_config in HDBSCAN_CONFIGS:
    for umap_config in UMAP_CONFIGS:
      params_options.append(
          {
              "embedding_config": emb_config,
              "hdbscan_config": hdb_config,
              "umap_config": umap_config,
              "compute_config": ComputeConfig(
                  # force_embeddings=True,
                  force_model=True
              ),
          }
      )
len(params_options)

1

In [30]:
# 3_333_616

In [34]:
for params in params_options:
  print("====="*40, flush=True)
  try:
    topic_model = model_builder.BerTopicModelBuilder(
        **base_params,
        **params
    )
    topic_model.fit_transform()
  except Exception as ex:
    print(f"[ERROR] Error running Bertopic: {ex}")

2026-08-24 06:23:03,962 | INFO | utils.bertopic_model_builder | [BerTopicModelBuilder] Starting fit_transform.
2026-08-24 06:23:03,969 | INFO | utils.bertopic_model_builder | [BerTopicModelBuilder] Loading embedding model: sentence-transformers/paraphrase-multilingual-mpnet-base-v2


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

2026-08-24 06:23:12,135 | INFO | utils.bertopic_model_builder | [BerTopicModelBuilder] Embedding model device: cuda
2026-08-24 06:23:12,142 | INFO | utils.bertopic_model_builder | [BerTopicModelBuilder] Enabled fp16 for embedding encode.
2026-08-24 06:23:12,143 | INFO | utils.bertopic_model_builder | [BerTopicModelBuilder] Embeddings cache not found. Computing embeddings.
2026-08-24 06:23:12,143 | INFO | utils.bertopic_model_builder | [BerTopicModelBuilder] Document representation: chunks
2026-08-24 06:23:12,145 | INFO | utils.bertopic_model_builder | [BerTopicModelBuilder] Loading chunk cache from disk.
2026-08-24 06:23:17,781 | INFO | utils.bertopic_model_builder | [BerTopicModelBuilder] Loaded 3333616 chunks.
2026-08-24 06:23:17,845 | INFO | utils.bertopic_model_builder | [BerTopicModelBuilder] Encoding 3333616 chunks.
2026-08-24 06:23:17,848 | INFO | utils.bertopic_model_builder | [BerTopicModelBuilder] Resuming embeddings from row 91136/3333616.
2026-08-24 06:23:17,852 | INFO | ut

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

2026-08-24 06:32:06,648 | INFO | utils.bertopic_model_builder | [BerTopicModelBuilder] Encoding batch 107520:123904 of 3333616.


...

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

2026-08-24 06:53:15,146 | INFO | utils.bertopic_model_builder | [BerTopicModelBuilder] Encoding batch 3318784:3333616 of 3333616.


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

2026-08-24 06:53:25,462 | INFO | utils.bertopic_model_builder | [BerTopicModelBuilder] Using chunk-level embeddings without pooling.
2026-08-24 06:57:05,361 | INFO | utils.bertopic_model_builder | [BerTopicModelBuilder] Saved embeddings cache: embeddings_paraphrase_multilingual_mpnet_base_v2_chunks_80.npy
2026-08-24 06:57:06,197 | INFO | utils.bertopic_model_builder | [BerTopicModelBuilder] Removed embedding checkpoint: embeddings_paraphrase_multilingual_mpnet_base_v2_chunks_80.partial.npy
2026-08-24 06:57:06,200 | INFO | utils.bertopic_model_builder | [BerTopicModelBuilder] Removed embedding checkpoint: embeddings_paraphrase_multilingual_mpnet_base_v2_chunks_80.progress.json
2026-08-24 06:57:06,201 | INFO | utils.bertopic_model_builder | [BerTopicModelBuilder] Prepared 3333616 texts and embeddings with shape (3333616, 768).
2026-08-24 06:57:06,201 | INFO | utils.bertopic_model_builder | [BerTopicModelBuilder] Initializing BERTopic model with configured UMAP and HDBSCAN.
2026-08-24 06:

KeyboardInterrupt: 

In [ ]:
generate_metrics_comparison_graphs(base_path=f"{BASE_PATH}/{FOLDER_NAME}")

{'base_path': '/content/drive/MyDrive/tesis/bertopic_models/senate',
 'runs_found': 38,
 'summary_csv': '/content/drive/MyDrive/tesis/bertopic_models/senate/metrics_comparison/comparison_summary.csv',
 'table_html': '/content/drive/MyDrive/tesis/bertopic_models/senate/metrics_comparison/comparison_table.html',
 'table_markdown': '/content/drive/MyDrive/tesis/bertopic_models/senate/metrics_comparison/comparison_table.md',
 'plots': {'coherence_c_v': '/content/drive/MyDrive/tesis/bertopic_models/senate/metrics_comparison/compare_coherence_c_v.png',
  'coherence_u_mass': '/content/drive/MyDrive/tesis/bertopic_models/senate/metrics_comparison/compare_coherence_u_mass.png',
  'coherence_c_npmi': '/content/drive/MyDrive/tesis/bertopic_models/senate/metrics_comparison/compare_coherence_c_npmi.png',
  'silhouette_score': '/content/drive/MyDrive/tesis/bertopic_models/senate/metrics_comparison/compare_silhouette_score.png',
  'topic_diversity': '/content/drive/MyDrive/tesis/bertopic_models/senat

## Topic Reduction

In [ ]:
TOP_CONFIGS = [
    {
        "embedding_config": EmbeddingConfig(
            embedding_model="sentence-transformers/paraphrase-multilingual-mpnet-base-v2",
            max_words=80,
            spacy_model="es_core_news_lg",
            document_representation=DocumentRepresentation.CHUNKS,
        ),
        "hdbscan_config": HDBSCANConfig(min_cluster_size=20, prediction_data=True),
        "umap_config": UMAPConfig(n_neighbors=15, metric="cosine", min_dist=0.1)
    },
    {
        "embedding_config": EmbeddingConfig(
            embedding_model="sentence-transformers/paraphrase-multilingual-mpnet-base-v2",
            max_words=80,
            spacy_model="es_core_news_lg",
            document_representation=DocumentRepresentation.CHUNKS,
        ),
        "hdbscan_config": HDBSCANConfig(min_cluster_size=20, prediction_data=True),
        "umap_config": UMAPConfig(n_neighbors=20, metric="cosine", min_dist=0.1)
    },
    {
        "embedding_config": EmbeddingConfig(
            embedding_model="sentence-transformers/paraphrase-multilingual-mpnet-base-v2",
            max_words=80,
            spacy_model="es_core_news_lg",
            document_representation=DocumentRepresentation.CHUNKS,
        ),
        "hdbscan_config": HDBSCANConfig(min_cluster_size=25, prediction_data=True),
        "umap_config": UMAPConfig(n_neighbors=15, metric="cosine", min_dist=0.1)
    },
    {
        "embedding_config": EmbeddingConfig(
            embedding_model="sentence-transformers/paraphrase-multilingual-mpnet-base-v2",
            max_words=80,
            spacy_model="es_core_news_lg",
            document_representation=DocumentRepresentation.CHUNKS,
        ),
        "hdbscan_config": HDBSCANConfig(min_cluster_size=25, prediction_data=True),
        "umap_config": UMAPConfig(n_neighbors=20, metric="cosine", min_dist=0.1)
    },
    {
        "embedding_config": EmbeddingConfig(
            embedding_model="sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2",
            max_words=80,
            spacy_model="es_core_news_lg",
            document_representation=DocumentRepresentation.CHUNKS,
        ),
        "hdbscan_config": HDBSCANConfig(min_cluster_size=20, prediction_data=True),
        "umap_config": UMAPConfig(n_neighbors=15, metric="cosine", min_dist=0.1)
    },

]

In [ ]:
BERTOPIC_MODELS = []
for config in TOP_CONFIGS:
  print("======="*10)
  bertopic = model_builder.BerTopicModelBuilder(
      **base_params,
      **config
  )
  bertopic.fit_transform()

  # save object reference
  BERTOPIC_MODELS.append(bertopic)

  # compute hierarchical topics
  hierarchical_topics = bertopic.get_hierarchical_topic(force_compute=True)
  tree = bertopic.topic_model.get_topic_tree(hierarchical_topics)

  trees_path = f"{bertopic._base_output_path}/outputs/hierarchical_tree"
  if not os.path.exists(trees_path):
    os.makedirs(trees_path)

  tree_file = f"{trees_path}/{bertopic.model_id}.txt"
  print(f"Saving tree file to: {tree_file}")
  with open(tree_file, "w+") as f:
    f.write(str(tree))

2026-06-08 02:35:31,449 | INFO | utils.bertopic_model_builder | [BerTopicModelBuilder] Starting fit_transform.
2026-06-08 02:35:31,450 | INFO | utils.bertopic_model_builder | [BerTopicModelBuilder] Loading embedding model: sentence-transformers/paraphrase-multilingual-mpnet-base-v2


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/5.12k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/9.08M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

2026-06-08 02:37:27,675 | INFO | utils.bertopic_model_builder | [BerTopicModelBuilder] Loading embeddings cache: embeddings_paraphrase_multilingual_minilm_l12_v2_chunks_80.npy
2026-06-08 02:37:29,097 | INFO | utils.bertopic_model_builder | [BerTopicModelBuilder] Embeddings loaded with shape (13895, 384).
2026-06-08 02:37:29,099 | INFO | utils.bertopic_model_builder | [BerTopicModelBuilder] Loading chunk cache from disk.
2026-06-08 02:37:29,129 | INFO | utils.bertopic_model_builder | [BerTopicModelBuilder] Loaded 13895 chunks.
2026-06-08 02:37:29,130 | INFO | utils.bertopic_model_builder | [BerTopicModelBuilder] Prepared 13895 texts and embeddings with shape (13895, 384).
2026-06-08 02:37:37,833 | INFO | utils.bertopic_model_builder | [BerTopicModelBuilder] Loaded saved BERTopic model: /content/drive/MyDrive/tesis/bertopic_models/senate/runs/emb_paraphrase_multilingual_minilm_l12_v2__rep_chunks__mw_80__umap_n15_c10__hdb_mcs20/bertopic_model
2026-06-08 02:37:38,878 | INFO | utils.bertopi

Saving tree file to: /content/drive/MyDrive/tesis/bertopic_models/senate/outputs/hierarchical_tree/emb_paraphrase_multilingual_minilm_l12_v2__rep_chunks__mw_80__umap_n15_c10__hdb_mcs20.txt


In [ ]:
BERTOPIC_MODELS_DICT = {
    model.model_id: model for model in BERTOPIC_MODELS
}

In [ ]:
merge_lists_folder = f"{BASE_PATH}/{FOLDER_NAME}/inputs/merge_trees"
MERGED_MODELS = []
for file_name in os.listdir(merge_lists_folder):
  print("========="*10)
  model_id = file_name.split(".")[0]
  print(model_id)

  if model_id in BERTOPIC_MODELS_DICT:
    bertopic = BERTOPIC_MODELS_DICT[model_id]

    # build merged topic lists
    merged_tree_file = f"{merge_lists_folder}/{file_name}"
    merge_topic_list = build_merge_topic_list_from_file(merged_tree_file)

    merged_model = bertopic.merge_topics(merge_topic_list)

    MERGED_MODELS.append(merged_model)

    print(f"Merged model coherence score: {merged_model.coherence_score}")
    print(f"Merged model total topics: {len(merged_model.topic_model.get_topics())}")



emb_paraphrase_multilingual_minilm_l12_v2__rep_chunks__mw_80__umap_n15_c10__hdb_mcs20


2026-06-08 03:32:34,569 - BERTopic - WARNING: When you use `pickle` to save/load a BERTopic model,please make sure that the environments in which you saveand load the model are **exactly** the same. The version of BERTopic,its dependencies, and python need to remain the same.
2026-06-08 03:32:38,160 | INFO | utils.bertopic_model_builder | [BerTopicModelBuilder] Saved BERTopic model: /content/drive/MyDrive/tesis/bertopic_models/senate/runs/emb_paraphrase_multilingual_minilm_l12_v2__rep_chunks__mw_80__umap_n15_c10__hdb_mcs20__merged/bertopic_model
2026-06-08 03:32:38,793 | INFO | utils.bertopic_model_builder | [BerTopicModelBuilder] Saved topics/probs artifacts: /content/drive/MyDrive/tesis/bertopic_models/senate/runs/emb_paraphrase_multilingual_minilm_l12_v2__rep_chunks__mw_80__umap_n15_c10__hdb_mcs20__merged/topics.npy, /content/drive/MyDrive/tesis/bertopic_models/senate/runs/emb_paraphrase_multilingual_minilm_l12_v2__rep_chunks__mw_80__umap_n15_c10__hdb_mcs20__merged/probs.npy
2026-06

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

2026-06-08 03:32:58,222 | INFO | utils.bertopic_model_builder | [BerTopicModelBuilder] Loading embeddings cache: embeddings_paraphrase_multilingual_minilm_l12_v2_chunks_80.npy
2026-06-08 03:32:58,254 | INFO | utils.bertopic_model_builder | [BerTopicModelBuilder] Embeddings loaded with shape (13895, 384).
2026-06-08 03:32:58,259 | INFO | utils.bertopic_model_builder | [BerTopicModelBuilder] Loading chunk cache from disk.
2026-06-08 03:32:58,280 | INFO | utils.bertopic_model_builder | [BerTopicModelBuilder] Loaded 13895 chunks.
2026-06-08 03:33:10,310 | INFO | utils.bertopic_model_builder | [BerTopicModelBuilder] Saved evaluation results to /content/drive/MyDrive/tesis/bertopic_models/senate/runs/emb_paraphrase_multilingual_minilm_l12_v2__rep_chunks__mw_80__umap_n15_c10__hdb_mcs20__merged/evaluation_metrics.json
2026-06-08 03:33:10,319 | INFO | utils.bertopic_model_builder | [BerTopicModelBuilder] Merged model persisted under: /content/drive/MyDrive/tesis/bertopic_models/senate/runs/emb_

Merged model coherence score: 0.6580930348987918
Merged model total topics: 34
emb_paraphrase_multilingual_mpnet_base_v2__rep_chunks__mw_80__umap_n20_c10__hdb_mcs20


...

Merged model coherence score: 0.6949246238378526
Merged model total topics: 43
emb_paraphrase_multilingual_mpnet_base_v2__rep_chunks__mw_80__umap_n15_c10__hdb_mcs25


2026-06-08 03:34:19,390 - BERTopic - WARNING: When you use `pickle` to save/load a BERTopic model,please make sure that the environments in which you saveand load the model are **exactly** the same. The version of BERTopic,its dependencies, and python need to remain the same.
2026-06-08 03:34:23,463 | INFO | utils.bertopic_model_builder | [BerTopicModelBuilder] Saved BERTopic model: /content/drive/MyDrive/tesis/bertopic_models/senate/runs/emb_paraphrase_multilingual_mpnet_base_v2__rep_chunks__mw_80__umap_n15_c10__hdb_mcs25__merged/bertopic_model
2026-06-08 03:34:24,089 | INFO | utils.bertopic_model_builder | [BerTopicModelBuilder] Saved topics/probs artifacts: /content/drive/MyDrive/tesis/bertopic_models/senate/runs/emb_paraphrase_multilingual_mpnet_base_v2__rep_chunks__mw_80__umap_n15_c10__hdb_mcs25__merged/topics.npy, /content/drive/MyDrive/tesis/bertopic_models/senate/runs/emb_paraphrase_multilingual_mpnet_base_v2__rep_chunks__mw_80__umap_n15_c10__hdb_mcs25__merged/probs.npy
2026-06

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

2026-06-08 03:34:30,490 | INFO | utils.bertopic_model_builder | [BerTopicModelBuilder] Loading embeddings cache: embeddings_paraphrase_multilingual_mpnet_base_v2_chunks_80.npy
2026-06-08 03:34:30,534 | INFO | utils.bertopic_model_builder | [BerTopicModelBuilder] Embeddings loaded with shape (13895, 768).
2026-06-08 03:34:30,536 | INFO | utils.bertopic_model_builder | [BerTopicModelBuilder] Loading chunk cache from disk.
2026-06-08 03:34:30,558 | INFO | utils.bertopic_model_builder | [BerTopicModelBuilder] Loaded 13895 chunks.
2026-06-08 03:34:43,842 | INFO | utils.bertopic_model_builder | [BerTopicModelBuilder] Saved evaluation results to /content/drive/MyDrive/tesis/bertopic_models/senate/runs/emb_paraphrase_multilingual_mpnet_base_v2__rep_chunks__mw_80__umap_n15_c10__hdb_mcs25__merged/evaluation_metrics.json
2026-06-08 03:34:43,846 | INFO | utils.bertopic_model_builder | [BerTopicModelBuilder] Merged model persisted under: /content/drive/MyDrive/tesis/bertopic_models/senate/runs/emb_

Merged model coherence score: 0.6766724682717211
Merged model total topics: 36


### Outlier reduction

In [ ]:
OUTLIER_REDUCTION_VARIANTS = [
    OutlierReductionConfig(
        enabled=True,
        strategy=OutlierReductionStrategy.CTFIDF,
        threshold=0.1,
    ),
    OutlierReductionConfig(
        enabled=True,
        strategy=OutlierReductionStrategy.PROBABILITIES,
        threshold=0.05,
    ),
    OutlierReductionConfig(
        enabled=True,
        strategy=OutlierReductionStrategy.DISTRIBUTIONS,
        threshold=0.05,
    )
]

In [ ]:
for merged_model in MERGED_MODELS[:1]:
  print("======"*10)
  print(merged_model.model_id)
  try:
    for outlier_config in OUTLIER_REDUCTION_VARIANTS:
      print("------"*10)
      print(outlier_config)
      or_result = merged_model.reduce_outliers(config=outlier_config)
      print(f"cv score: {or_result.coherence_score}")
  except Exception as ex:
    print(f"[ERROR] Error running outlier reduction: {ex}")


emb_paraphrase_multilingual_minilm_l12_v2__rep_chunks__mw_80__umap_n15_c10__hdb_mcs20__merged
------------------------------------------------------------
OutlierReductionConfig(enabled=True, strategy=<OutlierReductionStrategy.CTFIDF: 'c-tf-idf'>, threshold=0.1, distributions_params={})


2026-06-08 03:49:40,193 | INFO | utils.bertopic_model_builder | [BerTopicModelBuilder] Reducing outliers with strategy=c-tf-idf, threshold=0.1. Outlier assignments before reduction: 6508.
2026-06-08 03:49:40,482 | INFO | utils.bertopic_model_builder | [BerTopicModelBuilder] Outlier reduction complete. Reassigned 6072 assignments; outliers remaining: 436.
2026-06-08 03:49:40,485 - BERTopic - WARNING: Using a custom list of topic assignments may lead to errors if topic reduction techniques are used afterwards. Make sure that manually assigning topics is the last step in the pipeline.Note that topic embeddings will also be created through weightedc-TF-IDF embeddings instead of centroid embeddings.
2026-06-08 03:49:41,639 - BERTopic - WARNING: When you use `pickle` to save/load a BERTopic model,please make sure that the environments in which you saveand load the model are **exactly** the same. The version of BERTopic,its dependencies, and python need to remain the same.
2026-06-08 03:49:43

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

2026-06-08 03:49:50,989 | INFO | utils.bertopic_model_builder | [BerTopicModelBuilder] Loading embeddings cache: embeddings_paraphrase_multilingual_minilm_l12_v2_chunks_80.npy
2026-06-08 03:49:51,018 | INFO | utils.bertopic_model_builder | [BerTopicModelBuilder] Embeddings loaded with shape (13895, 384).
2026-06-08 03:49:51,020 | INFO | utils.bertopic_model_builder | [BerTopicModelBuilder] Loading chunk cache from disk.
2026-06-08 03:49:51,047 | INFO | utils.bertopic_model_builder | [BerTopicModelBuilder] Loaded 13895 chunks.
2026-06-08 03:50:07,977 | INFO | utils.bertopic_model_builder | [BerTopicModelBuilder] Saved evaluation results to /content/drive/MyDrive/tesis/bertopic_models/senate/runs/emb_paraphrase_multilingual_minilm_l12_v2__rep_chunks__mw_80__umap_n15_c10__hdb_mcs20__merged__or_ctfidf_t0p1/evaluation_metrics.json
2026-06-08 03:50:07,982 | INFO | utils.bertopic_model_builder | [BerTopicModelBuilder] Outlier-reduced model persisted under: /content/drive/MyDrive/tesis/bertopi

cv score: 0.703119246966706
------------------------------------------------------------
OutlierReductionConfig(enabled=True, strategy=<OutlierReductionStrategy.PROBABILITIES: 'probabilities'>, threshold=0.05, distributions_params={})


...

cv score: 0.6580930348987918
------------------------------------------------------------
OutlierReductionConfig(enabled=True, strategy=<OutlierReductionStrategy.DISTRIBUTIONS: 'distributions'>, threshold=0.05, distributions_params={})


2026-06-08 03:50:39,226 | INFO | utils.bertopic_model_builder | [BerTopicModelBuilder] Reducing outliers with strategy=distributions, threshold=0.05. Outlier assignments before reduction: 6508.
100%|██████████| 7/7 [00:06<00:00,  1.07it/s]
2026-06-08 03:50:45,808 | INFO | utils.bertopic_model_builder | [BerTopicModelBuilder] Outlier reduction complete. Reassigned 6501 assignments; outliers remaining: 7.
2026-06-08 03:50:45,813 - BERTopic - WARNING: Using a custom list of topic assignments may lead to errors if topic reduction techniques are used afterwards. Make sure that manually assigning topics is the last step in the pipeline.Note that topic embeddings will also be created through weightedc-TF-IDF embeddings instead of centroid embeddings.
2026-06-08 03:50:47,498 - BERTopic - WARNING: When you use `pickle` to save/load a BERTopic model,please make sure that the environments in which you saveand load the model are **exactly** the same. The version of BERTopic,its dependencies, and p

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

2026-06-08 03:50:56,894 | INFO | utils.bertopic_model_builder | [BerTopicModelBuilder] Loading embeddings cache: embeddings_paraphrase_multilingual_minilm_l12_v2_chunks_80.npy
2026-06-08 03:50:56,924 | INFO | utils.bertopic_model_builder | [BerTopicModelBuilder] Embeddings loaded with shape (13895, 384).
2026-06-08 03:50:56,926 | INFO | utils.bertopic_model_builder | [BerTopicModelBuilder] Loading chunk cache from disk.
2026-06-08 03:50:56,971 | INFO | utils.bertopic_model_builder | [BerTopicModelBuilder] Loaded 13895 chunks.
2026-06-08 03:51:15,045 | INFO | utils.bertopic_model_builder | [BerTopicModelBuilder] Saved evaluation results to /content/drive/MyDrive/tesis/bertopic_models/senate/runs/emb_paraphrase_multilingual_minilm_l12_v2__rep_chunks__mw_80__umap_n15_c10__hdb_mcs20__merged__or_distributions_t0p05/evaluation_metrics.json
2026-06-08 03:51:15,051 | INFO | utils.bertopic_model_builder | [BerTopicModelBuilder] Outlier-reduced model persisted under: /content/drive/MyDrive/tesis

cv score: 0.693516850522861


In [ ]:
generate_metrics_comparison_graphs(base_path=f"{BASE_PATH}/{FOLDER_NAME}")

{'base_path': '/content/drive/MyDrive/tesis/bertopic_models/senate',
 'runs_found': 50,
 'summary_csv': '/content/drive/MyDrive/tesis/bertopic_models/senate/metrics_comparison/comparison_summary.csv',
 'table_html': '/content/drive/MyDrive/tesis/bertopic_models/senate/metrics_comparison/comparison_table.html',
 'table_markdown': '/content/drive/MyDrive/tesis/bertopic_models/senate/metrics_comparison/comparison_table.md',
 'plots': {'coherence_c_v': '/content/drive/MyDrive/tesis/bertopic_models/senate/metrics_comparison/compare_coherence_c_v.png',
  'coherence_u_mass': '/content/drive/MyDrive/tesis/bertopic_models/senate/metrics_comparison/compare_coherence_u_mass.png',
  'coherence_c_npmi': '/content/drive/MyDrive/tesis/bertopic_models/senate/metrics_comparison/compare_coherence_c_npmi.png',
  'silhouette_score': '/content/drive/MyDrive/tesis/bertopic_models/senate/metrics_comparison/compare_silhouette_score.png',
  'topic_diversity': '/content/drive/MyDrive/tesis/bertopic_models/senat